# 03 · Train, compare, and evaluate

**Phases 4, 5 and 6** of the pipeline.

Trains one YOLOv8 model per dataset — the unprocessed baseline plus the nine
processed variants — then ranks them by validation mAP and evaluates the
winner once on the held-out test set.

This is the notebook that turns ten separate experiments into the single
comparison the assignment asks for.

## 1 · Setup

In [51]:
# ---------------------------------------------------------------------
# Locate the project root.
#
# This notebook may sit in notebook/, or in notebook/<yourname>/, so the
# search walks upwards until it finds the folder containing "data".
# Every path below is built from PROJECT_ROOT, so nothing breaks when the
# notebook is moved or when a teammate runs it on a different machine.
# ---------------------------------------------------------------------
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").is_dir():
    raise SystemExit(
        "Could not find the project root (a folder containing 'data').\n"
        "Set PROJECT_ROOT manually, for example:\n"
        "    PROJECT_ROOT = Path(r'C:/Users/you/PCB-Defect-Inspection')")

RAW_DIR       = PROJECT_ROOT / "data" / "raw"        # Annotations + images
YOLO_DIR      = PROJECT_ROOT / "data" / "yolo"       # generated in notebook 01
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"  # generated in notebook 02
RUNS_DIR      = PROJECT_ROOT / "runs" / "pcb_comparison"

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DIR, "  exists:", RAW_DIR.is_dir())

Project root : e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection
Raw data     : e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\raw   exists: True


In [52]:
import json

import pandas as pd

# ---------------------------------------------------------------------
# The baseline is an ordinary entry in this table, not a special case.
# Without it there is nothing to compare against, and a ranking of nine
# processed variants cannot show whether processing helped at all.
# ---------------------------------------------------------------------
EXPERIMENTS = {"BASE": YOLO_DIR / "data.yaml"}
for key in ["A1", "A2", "A3", "B1", "B2", "B3", "C1", "C2", "C3"]:
    EXPERIMENTS[key] = PROCESSED_DIR / f"pcb_{key}" / "data.yaml"

LABELS = {
    "BASE": "Baseline (no processing)",
    "A1": "Gaussian, light",
    "A2": "Gaussian, medium",
    "A3": "Gaussian, heavy",
    "B1": "Canny edge, weak",
    "B2": "Canny edge, medium",
    "B3": "Canny edge, strong",
    "C1": "Morphological, small",
    "C2": "Morphological, medium",
    "C3": "Morphological, large",
}

# ---------------------------------------------------------------------
# EVERY RUN USES THESE SAME SETTINGS.
#
# If one variant were given more epochs or a larger input size than
# another, the comparison would be measuring that difference instead of
# the image processing. The dataset is the only thing allowed to vary.
# ---------------------------------------------------------------------
MODEL    = "yolo11n.pt"
EPOCHS   = 75
IMGSZ    = 640   # the defects are only 2-4% of the image width, so 640
                  # may be too small to resolve them
BATCH    = 8      # lower this first if you run out of GPU memory
PATIENCE = 0     # 0 = early stopping disabled, so every run trains all 75 epochs
SEED     = 42     # same seed everywhere, so the comparison is not
                  # measuring random initialisation

# New output folder, so the YOLOv8n results survive intact for comparison.
RUNS_DIR = PROJECT_ROOT / "runs" / "yolo11_comparison"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'key':<6}{'dataset':<12}exists")
for key, path in EXPERIMENTS.items():
    print(f"{key:<6}{path.parent.name:<12}{path.exists()}")

key   dataset     exists
BASE  yolo        True
A1    pcb_A1      True
A2    pcb_A2      True
A3    pcb_A3      True
B1    pcb_B1      True
B2    pcb_B2      True
B3    pcb_B3      True
C1    pcb_C1      True
C2    pcb_C2      True
C3    pcb_C3      True


## 2 · Train

Ten runs. This is the long part — hours, not minutes.

Already-completed runs are skipped, so you can interrupt this cell and re-run
it later to resume where it stopped.

**Do a short trial first.** Set `EPOCHS = 3` above and run one configuration
to confirm the whole path works before committing to the real thing.

In [53]:
import itertools

for g, c in itertools.product(["A1", "A2", "A3"], ["B1", "B2", "B3"]):
    key = f"{g}{c}"
    EXPERIMENTS[key] = PROCESSED_DIR / f"pcb_{key}" / "data.yaml"
    LABELS[key] = f"Combined {g} + {c}"

print(len(EXPERIMENTS), "experiments:", list(EXPERIMENTS))

19 experiments: ['BASE', 'A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'A1B1', 'A1B2', 'A1B3', 'A2B1', 'A2B2', 'A2B3', 'A3B1', 'A3B2', 'A3B3']


In [54]:
from ultralytics import YOLO


def train_one(key, force=False):
    """Train a single configuration."""
    yaml_path = EXPERIMENTS[key]
    if not yaml_path.exists():
        print(f"SKIP {key}: {yaml_path} not found")
        return

    if (RUNS_DIR / key / "weights" / "best.pt").exists() and not force:
        print(f"SKIP {key}: already trained (pass force=True to redo)")
        return

    print(f"\n{'=' * 68}\nTRAINING {key}: {LABELS[key]}\n{'=' * 68}")

    # A fresh model every time. Continuing from a previous run's weights
    # would let one dataset influence the results of the next.
    model = YOLO(MODEL)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=PATIENCE, seed=SEED,
        project=str(RUNS_DIR), name=key, exist_ok=True,
        val=True,      # validate after every epoch
        plots=True,
    )


print("train_one() defined. Run the next cell to train everything.")

train_one() defined. Run the next cell to train everything.


In [29]:
# =====================================================================
# SEED STUDY - is A1's lead real, or run-to-run variation?
# =====================================================================
# The first comparison gave a spread of only 0.0233 mAP50-95 across ten
# configs, with a non-monotonic Canny group (B2 worse than both B1 and B3).
# That pattern suggests noise rather than a real effect. One seed per config
# measures a single draw; three draws per config separate the two.
#
# Four configurations are re-run: the baseline, plus the best of each
# technique family from the first pass.
# =====================================================================

SEED_CONFIGS = ["BASE", "A3", "B3", "C1"]
SEED_RUNS_DIR = RUNS_DIR
SEEDS = [0, 1]

# Full-length training with early stopping DISABLED.
# patience=0 is read by Ultralytics as infinity, so every run trains the
# same number of epochs. In the first pass A1 stopped at 47 while others
# ran 50, which is a small unfairness now removed.
SEED_EPOCHS = 75
SEED_PATIENCE = 0

SEED_RUNS_DIR = RUNS_DIR.parent / "seed_study"
SEED_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(SEED_CONFIGS)} configs x {len(SEEDS)} seeds "
      f"= {len(SEED_CONFIGS) * len(SEEDS)} runs at {SEED_EPOCHS} epochs")
print(f"Output: {SEED_RUNS_DIR}")

4 configs x 2 seeds = 8 runs at 75 epochs
Output: e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\seed_study


In [30]:
def train_seed(key, seed, force=False):
    """Train one configuration with one seed."""
    yaml_path = EXPERIMENTS[key]
    if not yaml_path.exists():
        print(f"SKIP {key}: dataset not found")
        return

    name = f"{key}_s{seed}"
    if (SEED_RUNS_DIR / name / "weights" / "best.pt").exists() and not force:
        print(f"SKIP {name}: already trained")
        return

    print(f"\n{'=' * 68}\n{name}: {LABELS[key]}, seed {seed}\n{'=' * 68}")

    model = YOLO(MODEL)
    model.train(
        data=str(yaml_path),
        epochs=SEED_EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=SEED_PATIENCE,   # 0 = no early stopping
        seed=seed,                # the ONLY thing that varies within a config
        project=str(SEED_RUNS_DIR), name=name, exist_ok=True,
        val=True, plots=False,    # plots off: 12 runs of plots is a lot of disk
    )


# Seed-major order: this gives you a complete set of four configs after
# each pass, so an interrupted study is still analysable.
for seed in SEEDS:
    for key in SEED_CONFIGS:
        train_seed(key, seed)

SKIP BASE_s0: already trained
SKIP A3_s0: already trained
SKIP B3_s0: already trained
SKIP C1_s0: already trained
SKIP BASE_s1: already trained
SKIP A3_s1: already trained
SKIP B3_s1: already trained
SKIP C1_s1: already trained


In [8]:
# Train the baseline first: if something is wrong, you find out on the
# simplest configuration rather than after nine runs.
for key in EXPERIMENTS:
    train_one(key)

SKIP BASE: already trained (pass force=True to redo)
SKIP A1: already trained (pass force=True to redo)
SKIP A2: already trained (pass force=True to redo)
SKIP A3: already trained (pass force=True to redo)
SKIP B1: already trained (pass force=True to redo)
SKIP B2: already trained (pass force=True to redo)
SKIP B3: already trained (pass force=True to redo)
SKIP C1: already trained (pass force=True to redo)
SKIP C2: already trained (pass force=True to redo)
SKIP C3: already trained (pass force=True to redo)

TRAINING A1B1: Combined A1 + B1
New https://pypi.org/project/ultralytics/8.4.140 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=Fal

In [34]:
for seed in [0, 1]:
    train_seed("A3", seed)


A3_s0: Gaussian, heavy, seed 0
New https://pypi.org/project/ultralytics/8.4.138 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\processed\pcb_A3\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x00000209BC5C0180>
Traceback (most recent call last):
  File "e:\Python 3.12.10\Lib\site-packages\torch\utils\data\dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "e:\Python 3.12.10\Lib\site-packages\torch\utils\data\dataloader.py", line 1562, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
                                   ^^^^^^^^^^^^^^^^^^^^
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      11/75      1.52G      2.124      2.033     0.9203          7        640: 100% ━━━━━━━━━━━━ 61/61 7.5it/s 8.1s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 7.3it/s 0.5s0.2s
                   all         60        303      0.784      0.756      0.816      0.396

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      12/75      1.52G       2.15      1.919     0.9183          6        640: 100% ━━━━━━━━━━━━ 61/61 7.9it/s 7.8s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 8.2it/s 0.5s0.2s
                   all         60        303      0.726      0.729      0.794      0.374

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      13/75      1.52G      2.084      1.863     0.9237          5        640: 100% ━━━

In [38]:
for seed in [0, 1]:
    train_seed("C1", seed)


C1_s0: Morphological, small, seed 0
New https://pypi.org/project/ultralytics/8.4.138 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\processed\pcb_C1\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, io

In [31]:
for seed in [0, 1]:
    train_seed("A3B2", seed)   # whichever cell won


A3B2_s0: Combined A3 + B2, seed 0
New https://pypi.org/project/ultralytics/8.4.140 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\processed\pcb_A3B2\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, io

In [55]:
# =====================================================================
# RETRAIN AT 100 EPOCHS FOR THE CROSS-MEMBER COMPARISON
# =====================================================================
# The 75-epoch study above is internally consistent and stands as the
# parameter selection work. These four configurations are retrained at the
# training budget agreed with the rest of the team, so that this member's
# best techniques can be compared against theirs on equal footing.
#
# A separate output folder keeps the 75-epoch results intact.
# =====================================================================

FINAL_CONFIGS = ["BASE", "A3", "B3", "A3B2"]   # <- your best combined cell
FINAL_EPOCHS = 100
FINAL_RUNS_DIR = PROJECT_ROOT / "runs" / "final_100ep"
FINAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Everything except epochs must match what the team agreed. Confirm these
# against your teammates' args.yaml before starting - a mismatch in imgsz
# or model would make the retraining pointless.
print(f"model {MODEL} | epochs {FINAL_EPOCHS} | imgsz {IMGSZ} | "
      f"batch {BATCH} | patience {PATIENCE} | seed {SEED}")
print("output:", FINAL_RUNS_DIR)

for key in FINAL_CONFIGS:
    print(f"  {key:6s} dataset exists: {EXPERIMENTS[key].exists()}")

model yolo11n.pt | epochs 100 | imgsz 640 | batch 8 | patience 0 | seed 42
output: e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\final_100ep
  BASE   dataset exists: True
  A3     dataset exists: True
  B3     dataset exists: True
  A3B2   dataset exists: True


In [56]:
def train_final(key, force=False):
    """Train one configuration at the agreed 100-epoch budget."""
    yaml_path = EXPERIMENTS[key]
    if not yaml_path.exists():
        print(f"SKIP {key}: dataset not found at {yaml_path}")
        return

    if (FINAL_RUNS_DIR / key / "weights" / "best.pt").exists() and not force:
        print(f"SKIP {key}: already trained")
        return

    print(f"\n{'=' * 68}\n{key}: {LABELS[key]} @ {FINAL_EPOCHS} epochs\n{'=' * 68}")

    model = YOLO(MODEL)
    model.train(
        data=str(yaml_path),
        epochs=FINAL_EPOCHS,
        imgsz=IMGSZ, batch=BATCH,
        patience=PATIENCE,        # 0 = no early stopping, so all runs match
        seed=SEED,
        project=str(FINAL_RUNS_DIR), name=key, exist_ok=True,
        val=True, plots=True,
    )


# BASE first: if something is wrong, you find out on the simplest config.
for key in FINAL_CONFIGS:
    train_final(key)


BASE: Baseline (no processing) @ 100 epochs
New https://pypi.org/project/ultralytics/8.4.140 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\yolo\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0

In [57]:
records = []
for key in FINAL_CONFIGS:
    csv = FINAL_RUNS_DIR / key / "results.csv"
    if not csv.exists():
        print(f"missing: {key}")
        continue
    frame = pd.read_csv(csv)
    frame.columns = [c.strip() for c in frame.columns]
    best = frame.loc[frame["metrics/mAP50-95(B)"].idxmax()]
    records.append({"key": key, "configuration": LABELS[key],
                    "mAP50-95": round(float(best["metrics/mAP50-95(B)"]), 4),
                    "mAP50": round(float(best["metrics/mAP50(B)"]), 4),
                    "precision": round(float(best["metrics/precision(B)"]), 4),
                    "recall": round(float(best["metrics/recall(B)"]), 4),
                    "best epoch": int(best["epoch"])})

final_table = pd.DataFrame(records).sort_values("mAP50-95", ascending=False)
final_table.to_csv(FINAL_RUNS_DIR / "final_comparison.csv", index=False)
print(final_table.to_string(index=False))

# Did 100 epochs actually help? Compare against the 75-epoch results.
for key in FINAL_CONFIGS:
    old = table[table["key"] == key]
    new = final_table[final_table["key"] == key]
    if len(old) and len(new):
        o, n = old["mAP50-95"].iat[0], new["mAP50-95"].iat[0]
        print(f"{key:6s} 75ep {o:.4f} -> 100ep {n:.4f}  ({n - o:+.4f})")

 key            configuration  mAP50-95  mAP50  precision  recall  best epoch
  A3          Gaussian, heavy    0.5313 0.9831     0.9960  0.9017          51
A3B2         Combined A3 + B2    0.5277 0.9784     0.9853  0.9179          48
  B3       Canny edge, strong    0.5168 0.9782     0.9465  0.9487          85
BASE Baseline (no processing)    0.5130 0.9757     0.9557  0.8808          88
BASE   75ep 0.4780 -> 100ep 0.5130  (+0.0350)
A3     75ep 0.5206 -> 100ep 0.5313  (+0.0107)
B3     75ep 0.4943 -> 100ep 0.5168  (+0.0225)
A3B2   75ep 0.5142 -> 100ep 0.5277  (+0.0135)


In [70]:
# =====================================================================
# REPORT ARTEFACTS - CSV and PNG, collected in one folder
# =====================================================================
import matplotlib.pyplot as plt

REPORT_DIR = FINAL_RUNS_DIR / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ---- collect ---------------------------------------------------------
records = []
for key in FINAL_CONFIGS:
    csv = FINAL_RUNS_DIR / key / "results.csv"
    if not csv.exists():
        print(f"missing: {key}")
        continue
    frame = pd.read_csv(csv)
    frame.columns = [c.strip() for c in frame.columns]
    # Best epoch, not last - this matches the checkpoint saved as best.pt.
    best = frame.loc[frame["metrics/mAP50-95(B)"].idxmax()]
    records.append({
        "key": key, "configuration": LABELS[key],
        "mAP50-95": round(float(best["metrics/mAP50-95(B)"]), 4),
        "mAP50": round(float(best["metrics/mAP50(B)"]), 4),
        "precision": round(float(best["metrics/precision(B)"]), 4),
        "recall": round(float(best["metrics/recall(B)"]), 4),
        "best epoch": int(best["epoch"]),
        "epochs run": int(frame["epoch"].max()),
    })

final_table = pd.DataFrame(records).sort_values("mAP50-95", ascending=False)
final_table.to_csv(REPORT_DIR / "final_comparison.csv", index=False)

# ---- table PNG -------------------------------------------------------
table_to_png(final_table, REPORT_DIR / "final_comparison_table.png",
             title="Final comparison at 100 epochs")

# ---- chart PNG -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

ordered = final_table.sort_values("mAP50-95")
colours = ["tab:orange" if k == "BASE" else "tab:blue" for k in ordered["key"]]
axes[0].barh(ordered["configuration"], ordered["mAP50-95"], color=colours)
axes[0].set_xlabel("mAP50-95 (validation)")
axes[0].set_title("Ranked results (orange = baseline)")
axes[0].grid(axis="x", alpha=0.3)

baseline = final_table.loc[final_table["key"] == "BASE", "mAP50-95"]
if len(baseline):
    axes[0].axvline(baseline.iat[0], color="tab:orange", linestyle="--", alpha=0.7)

for key in FINAL_CONFIGS:
    csv = FINAL_RUNS_DIR / key / "results.csv"
    if csv.exists():
        frame = pd.read_csv(csv)
        frame.columns = [c.strip() for c in frame.columns]
        axes[1].plot(frame["epoch"], frame["metrics/mAP50-95(B)"],
                     label=key, linewidth=2 if key == "BASE" else 1)

axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mAP50-95")
axes[1].set_title("Validation curves")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / "final_comparison_chart.png", dpi=200, bbox_inches="tight")
plt.show()

print(final_table.to_string(index=False))
print("\nSaved to", REPORT_DIR)

saved e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\final_100ep\report\final_comparison_table.png


<Figure size 1500x450 with 2 Axes>

 key            configuration  mAP50-95  mAP50  precision  recall  best epoch  epochs run
  A3          Gaussian, heavy    0.5313 0.9831     0.9960  0.9017          51         100
A3B2         Combined A3 + B2    0.5277 0.9784     0.9853  0.9179          48         100
  B3       Canny edge, strong    0.5168 0.9782     0.9465  0.9487          85         100
BASE Baseline (no processing)    0.5130 0.9757     0.9557  0.8808          88         100

Saved to e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\final_100ep\report


In [71]:
for key in FINAL_CONFIGS:
    old = table[table["key"] == key]
    new = final_table[final_table["key"] == key]
    if len(old) and len(new):
        o, n = old["mAP50-95"].iat[0], new["mAP50-95"].iat[0]
        print(f"{key:6s} 75ep {o:.4f} -> 100ep {n:.4f}  ({n - o:+.4f})")

BASE   75ep 0.4780 -> 100ep 0.5130  (+0.0350)
A3     75ep 0.5206 -> 100ep 0.5313  (+0.0107)
B3     75ep 0.4943 -> 100ep 0.5168  (+0.0225)
A3B2   75ep 0.5142 -> 100ep 0.5277  (+0.0135)


## 3 · Collect the results

Ultralytics writes one row per epoch to `results.csv`. The row taken here is
the **best** epoch by mAP50-95, not the last one, which matches the checkpoint
saved as `best.pt`. Reporting the last epoch would penalise any run that
overfitted slightly towards the end.

In [58]:
import yaml

for label, folder in [("stage 1", RUNS_DIR), ("stage 2", SEED_RUNS_DIR)]:
    print(f"\n=== {label}: {folder}")
    if not folder.exists():
        print("   (does not exist)")
        continue
    for run in sorted(p for p in folder.iterdir() if p.is_dir()):
        done = (run / "weights" / "best.pt").exists()
        args_file = run / "args.yaml"
        if args_file.exists():
            a = yaml.safe_load(args_file.read_text())
            print(f"   {run.name:12s} done={done}  model={a.get('model')}  "
                  f"epochs={a.get('epochs')}  imgsz={a.get('imgsz')}  "
                  f"batch={a.get('batch')}  seed={a.get('seed')}")
        else:
            print(f"   {run.name:12s} done={done}  (no args.yaml)")

print("\nSEED_CONFIGS is currently:", SEED_CONFIGS)


=== stage 1: e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\yolo11_comparison
   A1           done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A1B1         done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A1B2         done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A1B3         done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A2           done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A2B1         done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A2B2         done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A2B3         done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A3           done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A3_test      done=False  (no args.yaml)
   A3B1         done=True  model=yolo11n.pt  epochs=75  imgsz=640  batch=8  seed=42
   A3B2         done=Tr

In [59]:
import numpy as np
import pandas as pd

# Where stage 1 (seed 42) runs live, vs where stage 2 (seeds 0, 1) live.
STAGE1_DIR = RUNS_DIR
STAGE2_DIR = SEED_RUNS_DIR

SEED_CONFIGS = ["BASE", "A3", "B3", "C1", "A3B2"]   # set explicitly, right here
ALL_SEEDS = [42, 0, 1]


def run_folder(key, seed):
    """Stage 1 used the plain config name; stage 2 appends _s<seed>."""
    if seed == 42:
        return STAGE1_DIR / key
    return STAGE2_DIR / f"{key}_s{seed}"


records, missing = [], []
for key in SEED_CONFIGS:
    for seed in ALL_SEEDS:
        results_csv = run_folder(key, seed) / "results.csv"
        if not results_csv.exists():
            missing.append(f"{key} seed {seed} -> {results_csv}")
            continue
        frame = pd.read_csv(results_csv)
        frame.columns = [c.strip() for c in frame.columns]
        best = frame.loc[frame["metrics/mAP50-95(B)"].idxmax()]
        records.append({"key": key, "seed": seed,
                        "mAP50-95": round(float(best["metrics/mAP50-95(B)"]), 4),
                        "mAP50": round(float(best["metrics/mAP50(B)"]), 4),
                        "best epoch": int(best["epoch"])})

if missing:
    print("MISSING RUNS:")
    for m in missing:
        print("  ", m)

seed_results = pd.DataFrame(records)
if seed_results.empty:
    raise SystemExit("No runs found — check the paths printed above.")

summary = (seed_results.groupby("key")["mAP50-95"]
           .agg(["mean", "std", "min", "max", "count"])
           .round(4).sort_values("mean", ascending=False))
print("\n", summary.to_string())

if (summary["count"] < 3).any():
    print("\nNote: some configs have fewer than 3 seeds. A std from 2 samples "
          "is a weak estimate — treat the verdict below as provisional.")

    summary = (seed_results.groupby("key")["mAP50-95"]
           .agg(["mean", "std", "min", "max", "count"])
           .round(4).sort_values("mean", ascending=False))
print("\n", summary.to_string())

# Save both: the per-run detail and the aggregated summary.
seed_results.to_csv(RUNS_DIR / "seed_study_runs.csv", index=False)
summary.to_csv(RUNS_DIR / "seed_study_summary.csv")
print("\nSaved to", RUNS_DIR)


         mean     std     min     max  count
key                                        
A3B2  0.5153  0.0022  0.5138  0.5178      3
A3    0.5055  0.0133  0.4954  0.5206      3
B3    0.4961  0.0029  0.4943  0.4994      3
C1    0.4754  0.0075  0.4706  0.4841      3
BASE  0.4739  0.0076  0.4651  0.4785      3

         mean     std     min     max  count
key                                        
A3B2  0.5153  0.0022  0.5138  0.5178      3
A3    0.5055  0.0133  0.4954  0.5206      3
B3    0.4961  0.0029  0.4943  0.4994      3
C1    0.4754  0.0075  0.4706  0.4841      3
BASE  0.4739  0.0076  0.4651  0.4785      3

Saved to e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\yolo11_comparison


In [60]:
a3_mean, a3_std = summary.loc["A3", "mean"], summary.loc["A3", "std"]
cb_mean, cb_std = summary.loc["A3B2", "mean"], summary.loc["A3B2", "std"]

gap = cb_mean - a3_mean
pooled = float(np.sqrt(a3_std ** 2 + cb_std ** 2))

print(f"A3    : {a3_mean:.4f} +/- {a3_std:.4f}")
print(f"A3B2  : {cb_mean:.4f} +/- {cb_std:.4f}")
print(f"gap   : {gap:+.4f}  (pooled sd {pooled:.4f}, {abs(gap)/pooled:.1f} sigma)")
print("VERDICT:", "combining helps" if gap > 2 * pooled
      else "no benefit from combining")

A3    : 0.5055 +/- 0.0133
A3B2  : 0.5153 +/- 0.0022
gap   : +0.0098  (pooled sd 0.0135, 0.7 sigma)
VERDICT: no benefit from combining


In [61]:
import numpy as np

records = []
SEED_CONFIGS = ["BASE", "A3", "B3", "C1"]
for key in SEED_CONFIGS:
    for seed in [42] + SEEDS:
        # seed 42 came from stage 1, under the plain config name
        name = key if seed == 42 else f"{key}_s{seed}"
        results_csv = SEED_RUNS_DIR / name / "results.csv"
        if not results_csv.exists():
            print(f"missing: {name}")
            continue
        frame = pd.read_csv(results_csv)
        frame.columns = [c.strip() for c in frame.columns]
        best = frame.loc[frame["metrics/mAP50-95(B)"].idxmax()]
        records.append({"key": key, "seed": seed,
                        "mAP50-95": round(float(best["metrics/mAP50-95(B)"]), 4),
                        "mAP50": round(float(best["metrics/mAP50(B)"]), 4),
                        "best epoch": int(best["epoch"])})

seed_results = pd.DataFrame(records)
seed_results.to_csv(SEED_RUNS_DIR / "seed_results.csv", index=False)

summary = (seed_results.groupby("key")["mAP50-95"]
           .agg(["mean", "std", "min", "max", "count"])
           .round(4).sort_values("mean", ascending=False))
print(summary.to_string())

# ---------------------------------------------------------------------
# THE TEST THAT MATTERS.
#
# A config's lead is only meaningful if it exceeds the variation caused by
# the seed alone. If the ranges overlap, the two are indistinguishable and
# no winner can be claimed.
# ---------------------------------------------------------------------
if "BASE" in summary.index and len(summary) > 1:
    base_mean = summary.loc["BASE", "mean"]
    base_std = summary.loc["BASE", "std"]
    print(f"\nBaseline: {base_mean:.4f} +/- {base_std:.4f}")

    for key in summary.index:
        if key == "BASE":
            continue
        mean, std = summary.loc[key, "mean"], summary.loc[key, "std"]
        gap = mean - base_mean
        pooled = float(np.sqrt(base_std ** 2 + std ** 2))
        verdict = ("distinguishable" if abs(gap) > 2 * pooled
                   else "within seed noise")
        print(f"{key}: {mean:.4f} +/- {std:.4f}   "
              f"gap {gap:+.4f}   ({verdict})")

missing: BASE
missing: A3
missing: B3
missing: C1
        mean     std     min     max  count
key                                        
A3    0.4980  0.0037  0.4954  0.5006      2
B3    0.4970  0.0034  0.4946  0.4994      2
C1    0.4778  0.0088  0.4716  0.4841      2
BASE  0.4718  0.0095  0.4651  0.4785      2

Baseline: 0.4718 +/- 0.0095
A3: 0.4980 +/- 0.0037   gap +0.0262   (distinguishable)
B3: 0.4970 +/- 0.0034   gap +0.0252   (distinguishable)
C1: 0.4778 +/- 0.0088   gap +0.0060   (within seed noise)


In [62]:
def collect():
    rows = []
    for key in EXPERIMENTS:
        results_csv = RUNS_DIR / key / "results.csv"
        if not results_csv.exists():
            print(f"  no results yet for {key}")
            continue

        frame = pd.read_csv(results_csv)
        frame.columns = [c.strip() for c in frame.columns]

        metric = "metrics/mAP50-95(B)"
        if metric not in frame.columns:
            print(f"  unexpected columns in {results_csv}")
            continue

        best = frame.loc[frame[metric].idxmax()]
        rows.append({
            "key": key,
            "configuration": LABELS[key],
            "mAP50-95": round(float(best[metric]), 4),
            "mAP50": round(float(best["metrics/mAP50(B)"]), 4),
            "precision": round(float(best["metrics/precision(B)"]), 4),
            "recall": round(float(best["metrics/recall(B)"]), 4),
            "best epoch": int(best["epoch"]),
            "epochs run": int(frame["epoch"].max()),
        })

    if not rows:
        raise SystemExit(
            "No results found. Train the models first (section 2).\n"
            f"Looked in: {RUNS_DIR}")

    return pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)


table = collect()
table.to_csv(RUNS_DIR / "comparison.csv", index=False)
table

,key,configuration,mAP50-95,mAP50,precision,recall,best epoch,epochs run
3,A3,"Gaussian, heavy",0.5206,0.9804,0.9863,0.9157,58,75
17,A3B2,Combined A3 + B2,0.5142,0.9836,0.9882,0.9338,58,75
12,A1B3,Combined A1 + B3,0.5073,0.9575,0.8778,0.9329,74,75
13,A2B1,Combined A2 + B1,0.5073,0.9828,0.9981,0.8516,45,75
16,A3B1,Combined A3 + B1,0.5061,0.9607,0.9585,0.8081,42,75
10,A1B1,Combined A1 + B1,0.4989,0.9625,0.8548,0.9599,42,75
6,B3,"Canny edge, strong",0.4943,0.9642,0.9609,0.8933,65,75
5,B2,"Canny edge, medium",0.4918,0.9480,0.9543,0.9078,68,75
11,A1B2,Combined A1 + B2,0.4913,0.9379,0.8799,0.8521,37,75
18,A3B3,Combined A3 + B3,0.4912,0.9465,0.8412,0.8952,47,75


In [63]:
# ---------------------------------------------------------------------
# State the outcome plainly, including the awkward one.
# ---------------------------------------------------------------------
if "BASE" not in table["key"].values:
    print("WARNING: the baseline was not trained, so there is nothing to "
          "compare the processed variants against.")
else:
    baseline = table.loc[table["key"] == "BASE", "mAP50-95"].iat[0]
    best = table.iloc[0]

    print(f"Baseline mAP50-95 : {baseline:.4f}")
    print(f"Best variant      : {best['key']} ({best['configuration']}) "
          f"{best['mAP50-95']:.4f}")

    if best["key"] == "BASE":
        # A legitimate outcome that must be reported rather than hidden.
        # A technique that does not help is still a finding, and explaining
        # why belongs in the discussion.
        print("\nNo processed variant beat the baseline. Report this as it "
              "stands: on this dataset the processing did not improve "
              "detection.")
    else:
        change = 100 * (best["mAP50-95"] - baseline) / baseline
        print(f"\nBest variant improves on the baseline by {change:+.1f}% "
              f"relative.")

    beat = table[(table["mAP50-95"] > baseline) & (table["key"] != "BASE")]
    print(f"{len(beat)} of {len(table) - 1} processed variants beat the "
          f"baseline.")

    BEST_KEY = best["key"]
    print(f"\nBEST_KEY = {BEST_KEY!r}")

Baseline mAP50-95 : 0.4780
Best variant      : A3 (Gaussian, heavy) 0.5206

Best variant improves on the baseline by +8.9% relative.
12 of 18 processed variants beat the baseline.

BEST_KEY = 'A3'


In [64]:
combined = table[table["key"].str.len() == 4]
print(combined[["key", "mAP50-95", "precision", "recall"]].to_string(index=False))
print(f"\nbest combined : {combined['mAP50-95'].max():.4f}")
print(f"A3 alone      : {table.loc[table['key'] == 'A3', 'mAP50-95'].iat[0]:.4f}")
print(f"B3 alone      : {table.loc[table['key'] == 'B3', 'mAP50-95'].iat[0]:.4f}")

 key  mAP50-95  precision  recall
A3B2    0.5142     0.9882  0.9338
A1B3    0.5073     0.8778  0.9329
A2B1    0.5073     0.9981  0.8516
A3B1    0.5061     0.9585  0.8081
A1B1    0.4989     0.8548  0.9599
A1B2    0.4913     0.8799  0.8521
A3B3    0.4912     0.8412  0.8952
BASE    0.4780     0.8748  0.8643
A2B2    0.4692     0.9699  0.8267
A2B3    0.4453     0.8357  0.7462

best combined : 0.5142
A3 alone      : 0.5206
B3 alone      : 0.4943


In [65]:
import numpy as np
grid = np.full((3, 3), np.nan)
for i, g in enumerate(["A1", "A2", "A3"]):
    for j, c in enumerate(["B1", "B2", "B3"]):
        row = table[table["key"] == f"{g}{c}"]
        if len(row):
            grid[i, j] = row["mAP50-95"].iat[0]
print(pd.DataFrame(grid, index=["A1","A2","A3"], columns=["B1","B2","B3"]).round(4))

        B1      B2      B3
A1  0.4989  0.4913  0.5073
A2  0.5073  0.4692  0.4453
A3  0.5061  0.5142  0.4912


## 4 · Chart for the report

In [66]:
import matplotlib.pyplot as plt

if "table" not in dir():
    raise SystemExit("Run the collect cell in section 3 first.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ordered = table.sort_values("mAP50-95")
colours = ["tab:orange" if k == "BASE" else "tab:blue" for k in ordered["key"]]

axes[0].barh(ordered["configuration"], ordered["mAP50-95"], color=colours)
axes[0].set_xlabel("mAP50-95 (validation)")
axes[0].set_title("Ranked results (orange = baseline)")
axes[0].grid(axis="x", alpha=0.3)

if "BASE" in table["key"].values:
    base_value = table.loc[table["key"] == "BASE", "mAP50-95"].iat[0]
    axes[0].axvline(base_value, color="tab:orange", linestyle="--", alpha=0.7)

for key in EXPERIMENTS:
    results_csv = RUNS_DIR / key / "results.csv"
    if results_csv.exists():
        frame = pd.read_csv(results_csv)
        frame.columns = [c.strip() for c in frame.columns]
        axes[1].plot(frame["epoch"], frame["metrics/mAP50-95(B)"],
                     label=key, linewidth=2 if key == "BASE" else 1)

axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mAP50-95")
axes[1].set_title("Validation curves")
axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RUNS_DIR / "comparison.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1500x500 with 2 Axes>

In [67]:
table.to_csv(RUNS_DIR / "comparison_full.csv", index=False)
summary.to_csv(RUNS_DIR / "seed_study_summary.csv")

In [68]:
import numpy as np
import pandas as pd

GAUSS = ["A1", "A2", "A3"]
CANNY = ["B1", "B2", "B3"]

grid = np.full((3, 3), np.nan)
for i, g in enumerate(GAUSS):
    for j, c in enumerate(CANNY):
        row = table[table["key"] == f"{g}{c}"]
        if len(row):
            grid[i, j] = row["mAP50-95"].iat[0]

grid_df = pd.DataFrame(grid, index=GAUSS, columns=CANNY).round(4)
print(grid_df.to_string())

        B1      B2      B3
A1  0.4989  0.4913  0.5073
A2  0.5073  0.4692  0.4453
A3  0.5061  0.5142  0.4912


In [69]:
grid_out = grid_df.reset_index().rename(columns={"index": "Gaussian"})

table_to_png(grid_out,
             RUNS_DIR / "grid_table.png",
             title="Gaussian x Canny interaction (validation mAP50-95)",
             highlight_col="B3",      # column scanned for the best value
             key_col="Gaussian",
             baseline_key=None)       # no baseline row in this table

saved e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\yolo11_comparison\grid_table.png


In [46]:
import pandas as pd
import matplotlib.pyplot as plt


def table_to_png(df, out_path,
                 title="Comparison of image processing configurations",
                 highlight_col="mAP50-95", baseline_key="BASE", key_col="key"):
    """Render a DataFrame as a PNG table.

    The baseline row is tinted orange and the best-scoring row green, so the
    two rows a reader looks for first are findable at a glance.
    """
    df = df.copy()
    n_rows, n_cols = df.shape
    text = df.values.astype(str)

    # Size each column to the longest string it must hold, otherwise long
    # labels such as "Baseline (no processing)" are silently clipped.
    widths = [max([len(str(col))] + [len(v) for v in text[:, j]])
              for j, col in enumerate(df.columns)]
    total = sum(widths)
    col_widths = [w / total for w in widths]

    fig, ax = plt.subplots(figsize=(max(total * 0.13, 9), 0.40 * n_rows + 1.0))
    ax.axis("off")

    table = ax.table(cellText=text, colLabels=df.columns,
                     colWidths=col_widths, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.45)

    best_row = df[highlight_col].astype(float).idxmax()

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if row == 0:
            cell.set_facecolor("#37474f")
            cell.set_text_props(color="white", weight="bold")
            continue
        if df.iloc[row - 1][key_col] == baseline_key:
            cell.set_facecolor("#ffe0b2")        # baseline
        elif df.index[row - 1] == best_row:
            cell.set_facecolor("#c8e6c9")        # best
        elif row % 2 == 0:
            cell.set_facecolor("#f5f5f5")

    ax.set_title(title, fontsize=13, weight="bold", pad=12)
    plt.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print("saved", out_path)


table_to_png(pd.read_csv(RUNS_DIR / "comparison.csv"),
             RUNS_DIR / "comparison_table.png")


saved e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\yolo11_comparison\comparison_table.png


In [47]:
# Seed study — reset_index() first, since summary has "key" as the index
table_to_png(summary.reset_index(),
             RUNS_DIR / "seed_study_table.png",
             title="Seed study: mean +/- std over 3 seeds",
             highlight_col="mean")

# The 3x3 grid, if you built it
table_to_png(grid_df.reset_index(),
             RUNS_DIR / "grid_table.png",
             title="Gaussian x Canny interaction (mAP50-95)",
             highlight_col="B3", key_col="index")

saved e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\yolo11_comparison\seed_study_table.png
saved e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\runs\yolo11_comparison\grid_table.png


## 5 · Final test evaluation

**This runs once, on the winner only, after all tuning is finished.**

The test set stays untouched during training and selection, which is what
makes this number an unbiased estimate. Evaluating every variant on the test
set and then picking the best would defeat that: the choice would have been
made using the test data, and the reported score would be optimistic.

This is the number that goes in your report.

In [51]:
BEST_KEY = "A3"

In [52]:
# BEST_KEY is set by the cell in section 3. Override it here if you want to
# evaluate a different configuration.
if "BEST_KEY" not in dir():
    raise SystemExit("Run section 3 first, or set BEST_KEY manually, "
                     "e.g. BEST_KEY = 'B2'")

weights = RUNS_DIR / BEST_KEY / "weights" / "best.pt"
if not weights.exists():
    raise SystemExit(f"No trained weights at {weights}")

print(f"FINAL TEST EVALUATION: {BEST_KEY} ({LABELS[BEST_KEY]})\n")

model = YOLO(str(weights))
metrics = model.val(data=str(EXPERIMENTS[BEST_KEY]), split="test",
                    project=str(RUNS_DIR), name=f"{BEST_KEY}_test",
                    exist_ok=True)

summary = {
    "configuration": BEST_KEY,
    "description": LABELS[BEST_KEY],
    "test mAP50-95": round(float(metrics.box.map), 4),
    "test mAP50": round(float(metrics.box.map50), 4),
    "test precision": round(float(metrics.box.mp), 4),
    "test recall": round(float(metrics.box.mr), 4),
}

print("Final held-out test result (report this):")
for name, value in summary.items():
    print(f"   {name:<18}{value}")

# Per-class results usually make the most useful discussion: they show
# whether one defect type is dragging the average down.
print("\nPer class (mAP50-95):")
for index, class_name in model.names.items():
    try:
        print(f"   {class_name:<18}{metrics.box.maps[index]:.4f}")
    except (IndexError, TypeError):
        pass

(RUNS_DIR / "test_result.json").write_text(json.dumps(summary, indent=2))
print(f"\nSaved to {RUNS_DIR / 'test_result.json'}")

FINAL TEST EVALUATION: A3 (Gaussian, heavy)

Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.40.0 ms, read: 89.819.9 MB/s, size: 878.2 KB)
val: Scanning E:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\processed\pcb_A3\labels\test... 152 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 152/152 489.6it/s 0.3s0.0s
val: New cache created: E:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\processed\pcb_A3\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.7it/s 5.7s.4ss
                   all        152        520      0.745      0.622      0.659      0.298
          missing_hole         25         85      0.995      0.953      0.969      0.532
            mouse_bite         25         86      0.942      0.68

---

## What to put in the report

- `runs/pcb_comparison/comparison.csv` — the ranked validation table
- `runs/pcb_comparison/comparison.png` — the ranking chart and curves
- `runs/pcb_comparison/test_result.json` — the final held-out number
- `runs/pcb_comparison/<KEY>/` — per-run curves, confusion matrix, sample
  predictions produced by Ultralytics

Report the negative results too. A configuration that made detection *worse*
is evidence about where a technique stops helping, and a comparative study
that reports only its successes is not a comparative study.

In [14]:
import pandas as pd
t = pd.read_csv(RUNS_DIR / "comparison.csv")
print(t.to_string(index=False))
print("\nspread:", round(t["mAP50-95"].max() - t["mAP50-95"].min(), 4))

 key            configuration  mAP50-95  mAP50  precision  recall  best epoch  epochs run
  A1          Gaussian, light    0.4796 0.9232     0.8817  0.8534          27          47
  C2    Morphological, medium    0.4693 0.9289     0.9456  0.7901          50          50
  B3       Canny edge, strong    0.4684 0.9333     0.9611  0.8091          36          50
  C3     Morphological, large    0.4677 0.9382     0.8899  0.8235          45          50
BASE Baseline (no processing)    0.4663 0.9475     0.8094  0.9282          50          50
  A2         Gaussian, medium    0.4643 0.9359     0.9855  0.7810          45          50
  B1         Canny edge, weak    0.4638 0.9209     0.8182  0.8794          32          50
  C1     Morphological, small    0.4638 0.9268     0.8965  0.8239          36          50
  A3          Gaussian, heavy    0.4600 0.9330     0.8796  0.8768          33          50
  B2       Canny edge, medium    0.4563 0.9097     0.8825  0.8104          49          50

spread: 0